# Графовая нейросеть для детекции ботов на Авито, прогон на GPU

Ноутбук берёт готовые признаки и модули из подключённого датасета, обучает GraphSAGE на графе кука–кука
по общим объявлениям с десятью сидами и сравнивает её с LightGBM и стеком на одних и тех же сплитах.

Перед запуском подключить датасет с содержимым `kaggle_bundle.zip` через Add Input и включить GPU
в настройках ноутбука (Settings, Accelerator). Интернет не нужен.

In [ ]:
import os, sys, shutil, glob
from pathlib import Path
import torch

os.environ["BOT_SEED"] = "11"
assert torch.cuda.is_available(), "GPU не включён: Settings, Accelerator, GPU"
print("GPU:", torch.cuda.get_device_name(0))

source = next(Path(p).parent for p in glob.glob("/kaggle/input/**/gnn_model.py", recursive=True))
work = Path("/kaggle/working/bot")
shutil.copytree(source, work, dirs_exist_ok=True)
os.chdir(work)
print("рабочая папка:", work)

## Обучение

Варианты `gnn` (два слоя GraphSAGE) и `gnn1` (один слой). Число эпох выбирает ранняя остановка на хронологическом
сплите, затем оно фиксируется для кросс-валидации. Логиты усредняются по десяти сидам. Около 5 минут.

In [ ]:
!{sys.executable} gnn_model.py --out out --variants gnn1,gnn --write --device cuda --seeds 11,22,33,44,55,66,77,88,99,100

In [ ]:
import pandas as pd
pd.read_csv("out/gnn_results.csv")

Как читать. `chrono_p_at_r70` получен обучением на неделе 0 и оценкой на неделе 1, `cv_folds_mean` и `cv_folds_std`
по кросс-валидации, две последние колонки показывают, в скольких фолдах модель обошла LightGBM и стек.

In [ ]:
for f in glob.glob("out/submission_*.csv"):
    shutil.copy(f, "/kaggle/working/" + Path(f).name)
print(sorted(p for p in os.listdir("/kaggle/working") if p.startswith("submission")))

Файлы предсказаний лежат в панели Output справа. Финальный файл называется `submission_gnn_stack.csv`,
это смесь рангов GraphSAGE и стека.